# InterFrames model gateway on Kaggle

This notebook loads weights from `models.zip` and backend code from `kaggle_setup.zip`, then exposes the three models under one authenticated temporary URL. The routes are `/moderation`, `/gma`, `/interframes`.

## Before you run it

1. Add a Kaggle input dataset containing `models.zip` and `kaggle_setup.zip` (or the unpacked `models/` and `backend/` folders).
2. `models.zip` needs `Interframes_Model/epoch14.keras`, `GMA_Model/gma.pth`, and `Moderation_Model/moderation.safetensors`. `kaggle_setup.zip` can zip either the repository root (`backend/...`) or the `backend/` folder (`app/...`), including `backend/app/models/gma_config/`, `backend/app/models/moderation_config/`, and `backend/app/models/interframes_protocol.json`.
3. Enable Internet in Kaggle Notebook settings. The setup cell installs missing API/adapter packages and verifies the model weights and config files.
4. Run the cells interactively. The last cell stays active to keep the temporary tunnel alive; do not use **Save & Run All** to host the service.

The adapters currently run inference on CPU. The public Cloudflare Quick Tunnel address is temporary and only works while this Kaggle session is running. Quick Tunnels are intended for testing, and anyone who gets the URL still needs the bearer token. See [Kaggle notebook settings](https://www.kaggle.com/docs/notebooks) and [Cloudflare Quick Tunnels](https://developers.cloudflare.com/cloudflare-one/connections/connect-networks/do-more-with-tunnels/trycloudflare/).

In [ ]:
from pathlib import Path, PurePosixPath
from zipfile import ZipFile
import os
import shutil
import subprocess
import sys

INPUT_ROOT = Path(os.environ.get("KAGGLE_INPUT_DIR", "/kaggle/input"))
PROJECT_DIR = Path("/kaggle/working/Interframes")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
model_dir = PROJECT_DIR / "models"
model_dir.mkdir(parents=True, exist_ok=True)
expected = (
    ("Interframes_Model", "epoch14.keras"),
    ("GMA_Model", "gma.pth"),
    ("Moderation_Model", "moderation.safetensors"),
)
def has_model_files(root):
    return all((root / Path(*parts)).is_file() for parts in expected)

# Discover inputs by their contents, so dataset-owner slugs and folder names can vary.
model_roots = {path.parent.parent for path in INPUT_ROOT.rglob("epoch14.keras")}
model_roots = {root for root in model_roots if has_model_files(root)}
model_archives = list(INPUT_ROOT.rglob("models.zip"))
if len(model_roots) == 1:
    model_root = next(iter(model_roots))
    for parts in expected:
        source = model_root / Path(*parts)
        destination = model_dir / Path(*parts)
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
elif len(model_archives) == 1:
    with ZipFile(model_archives[0]) as archive:
        members = [name for name in archive.namelist() if not name.endswith("/")]
        for parts in expected:
            suffix = tuple(parts)
            matches = [name for name in members if tuple(PurePosixPath(name.replace("\\", "/")).parts[-len(suffix):]) == suffix]
            if len(matches) != 1:
                member_label = "/".join(suffix)
                raise FileNotFoundError(f"Expected one {member_label} in models.zip; found {len(matches)}.")
            destination = model_dir / Path(*parts)
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(matches[0]) as source, destination.open("wb") as target:
                shutil.copyfileobj(source, target)
else:
    raise FileNotFoundError("Could not uniquely find all three model files under the Kaggle input directory.")

source_roots = set()
for gateway in INPUT_ROOT.rglob("model_gateway.py"):
    if gateway.parent.name == "app" and (gateway.parent / "models" / "gma_config" / "network.py").is_file():
        source_roots.add(gateway.parents[1])
source_archives = list(INPUT_ROOT.rglob("kaggle_setup.zip"))
backend_dir = PROJECT_DIR / "backend"
shutil.rmtree(backend_dir, ignore_errors=True)
if len(source_archives) == 1:
    staging_dir = Path("/kaggle/working/_kaggle_setup_extract")
    shutil.rmtree(staging_dir, ignore_errors=True)
    staging_dir.mkdir(parents=True, exist_ok=True)
    with ZipFile(source_archives[0]) as archive:
        for item in archive.infolist():
            relative = PurePosixPath(item.filename.replace("\\", "/"))
            destination = (staging_dir / Path(*relative.parts)).resolve()
            if staging_dir.resolve() not in destination.parents and destination != staging_dir.resolve():
                raise ValueError(f"Unsafe source archive path: {item.filename}")
            if item.is_dir():
                destination.mkdir(parents=True, exist_ok=True)
            else:
                destination.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(item) as source, destination.open("wb") as target:
                    shutil.copyfileobj(source, target)
    extracted_backends = {g.parents[1] for g in staging_dir.rglob("model_gateway.py") if g.parent.name == "app"}
    if len(extracted_backends) != 1:
        raise FileNotFoundError("Expected one app/model_gateway.py inside kaggle_setup.zip.")
    shutil.copytree(extracted_backends.pop(), backend_dir)
    shutil.rmtree(staging_dir, ignore_errors=True)
elif len(source_roots) == 1:
    shutil.copytree(source_roots.pop(), backend_dir)
else:
    raise FileNotFoundError("Could not find kaggle_setup.zip or backend/app/model_gateway.py under Kaggle inputs.")

gateway_file = backend_dir / "app" / "model_gateway.py"
models_pkg = backend_dir / "app" / "models"
required_support = (
    models_pkg / "interframes_protocol.json",
    models_pkg / "gma_config" / "network.py",
    models_pkg / "moderation_config" / "config.json",
    models_pkg / "moderation_config" / "preprocessor_config.json",
)
if not gateway_file.is_file() or not all(p.is_file() for p in required_support) or not has_model_files(model_dir):
    raise FileNotFoundError("Kaggle inputs must include the three model files and the backend/app/models config files.")
print("Found InterFrames source and all three model weights under the Kaggle input directory.")

In [ ]:
# Kaggle already includes PyTorch and TensorFlow; install the API and adapter dependencies.
import hashlib
packages = [
    "fastapi>=0.115,<1", "uvicorn[standard]>=0.30,<1", "python-dotenv>=1.0",
    "transformers>=4.44,<5", "safetensors>=0.4", "einops>=0.8",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

expected_hashes = {
    model_dir / "Interframes_Model" / "epoch14.keras": "8a5cde4b34ba5aa33fc407091665017cc42e7e54c58b8e2f2676243613045440",
    model_dir / "GMA_Model" / "gma.pth": "4a9faacbec5d095e019fcbba9e19871c9df9c097854f6617c7e376928c338149",
    model_dir / "Moderation_Model" / "moderation.safetensors": "97b2ce64ec146884b37f98ee7944ca4891aa72f6827dc0cb10684a1cbecd5830",
}
for weight_path, expected in expected_hashes.items():
    digest = hashlib.sha256(weight_path.read_bytes()).hexdigest()
    if digest != expected:
        raise RuntimeError(f"SHA-256 mismatch for {weight_path.name}: got {digest}")
print("Dependencies installed and all three model weights passed SHA-256 checks.")

## Set the shared secret

Generate a long random token on your computer (for example, with Python `secrets.token_urlsafe(32)`). Put it in the root `.env` as `KAGGLE_MODEL_SERVICE_TOKEN`, and enter the same value below. This input is hidden and the notebook never prints the token. Do not save the entered value in a notebook output.

In [ ]:
from getpass import getpass
import os
from pathlib import Path

PROJECT_DIR = Path("/kaggle/working/Interframes")
model_dir = PROJECT_DIR / "models"
gateway_file = PROJECT_DIR / "backend" / "app" / "model_gateway.py"
if not gateway_file.is_file() or not model_dir.is_dir():
    raise RuntimeError("Run the input-discovery/setup cell above successfully before entering the token.")

SERVICE_TOKEN = getpass("Enter the same random token as KAGGLE_MODEL_SERVICE_TOKEN in your local .env: " )
if len(SERVICE_TOKEN) < 32:
    raise ValueError("Use a token with at least 32 characters.")
os.environ["MODEL_SERVICE_TOKEN"] = SERVICE_TOKEN
os.environ["MODEL_DIR"] = str(model_dir)
os.environ["PYTHONPATH"] = str(PROJECT_DIR / "backend")
os.environ["GMA_SERVICE_URL"] = ""
os.environ["INTERFRAMES_SERVICE_URL"] = ""
os.environ["MODERATION_SERVICE_URL"] = ""
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["TF_NUM_INTRAOP_THREADS"] = "2"
os.environ["TF_NUM_INTEROP_THREADS"] = "2"
print("Token accepted; it was not displayed.")

In [ ]:
import time
from urllib.request import Request, urlopen

PORT = 8001
SERVER_LOG = Path("/kaggle/working/model-gateway.log")
server_env = os.environ.copy()
server_log = SERVER_LOG.open("w")
server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.model_gateway:app", "--app-dir", str(PROJECT_DIR / "backend"), "--host", "0.0.0.0", "--port", str(PORT)],
    cwd=PROJECT_DIR, env=server_env, stdout=server_log, stderr=subprocess.STDOUT,
)

headers = {"Authorization": f"Bearer {SERVICE_TOKEN}"}
for _ in range(1800):
    if server.poll() is not None:
        raise RuntimeError(SERVER_LOG.read_text(errors="replace")[-6000:])
    try:
        request = Request(f"http://127.0.0.1:{PORT}/moderation/health", headers=headers)
        with urlopen(request, timeout=5) as response:
            if response.status == 200:
                break
    except Exception:
        time.sleep(1)
else:
    raise TimeoutError("The three models did not finish loading within 30 minutes. Check model-gateway.log.")
print("All three model weights loaded and the authenticated local gateway is ready.")

In [ ]:
import queue
import re
import threading
import urllib.request
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

# Kaggle's working directory can be mounted without execute permission; use /tmp for the binary.
cloudflared = Path("/tmp/interframes-cloudflared")
if not cloudflared.exists():
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", cloudflared)
cloudflared.chmod(0o755)

# Stop a tunnel left running if this cell is being rerun in the same notebook session.
previous_tunnel = globals().get("tunnel")
if previous_tunnel is not None and previous_tunnel.poll() is None:
    previous_tunnel.terminate()
    try:
        previous_tunnel.wait(timeout=10)
    except subprocess.TimeoutExpired:
        previous_tunnel.kill()
        previous_tunnel.wait()

tunnel = subprocess.Popen(
    [str(cloudflared), "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
tunnel_lines = queue.Queue()
def read_tunnel_output():
    for line in tunnel.stdout:
        tunnel_lines.put(line.rstrip())
threading.Thread(target=read_tunnel_output, daemon=True).start()

gateway_url = None
url_pattern = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
deadline = time.time() + 120
while time.time() < deadline and tunnel.poll() is None:
    try:
        line = tunnel_lines.get(timeout=2)
        print(line)
        match = url_pattern.search(line)
        if match:
            gateway_url = match.group(0)
            break
    except queue.Empty:
        pass
if gateway_url is None:
    raise RuntimeError("Cloudflare did not provide a tunnel URL. Confirm Kaggle Internet is enabled.")

# Give the temporary hostname time to appear in DNS before checking each route.
for kind in ("moderation", "gma", "interframes"):
    last_error = None
    for attempt in range(15):
        request = Request(f"{gateway_url}/{kind}/health", headers=headers)
        try:
            with urlopen(request, timeout=15) as response:
                if response.status == 200:
                    break
        except HTTPError as exc:
            if not 500 <= exc.code < 600:
                raise RuntimeError(f"{kind} health check returned HTTP {exc.code}; check the token and model gateway.") from exc
            last_error = exc
        except URLError as exc:
            last_error = exc
            if tunnel.poll() is not None:
                raise RuntimeError("The Cloudflare tunnel stopped. Rerun the tunnel cell to get a live URL.") from exc
        if attempt < 14:
            time.sleep(3)
    else:
        raise RuntimeError(f"{kind} route is still unreachable at {gateway_url}. Confirm Kaggle Internet is enabled and the tunnel is active. Last error: {last_error}")

print("\nGateway origin (one URL):", gateway_url)
print("Copy this URL line into the root .env:")
print(f"KAGGLE_MODEL_GATEWAY_URL={gateway_url}")
print("Then set KAGGLE_MODEL_SERVICE_TOKEN in .env to the same hidden token you entered above.")
print("Model routes: /moderation, /gma, /interframes")

In [ ]:
# Keep this cell running so Kaggle does not treat the model server as idle.
try:
    while True:
        time.sleep(300)
        print("Kaggle model gateway is still running:", gateway_url)
except KeyboardInterrupt:
    tunnel.terminate()
    server.terminate()
    print("Stopped the tunnel and model gateway.")